# Reinforcement learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/advanced/04_reinforcement_learning.ipynb)

Learn a neural feedback law for a stochastic planning problem, then use it as an ordinary controller block.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Learn a policy for a problem

Reinforcement learning is a planner like the others: it reads a `PlanningProblem` and
returns a feedback law $u = \pi_\theta(x)$, here a small neural network. What it needs that
the others do not is a **stochastic** problem: a start drawn from a law, so that the law
learns from everywhere it will be used.

The task: swing up a pendulum whose motor ($|u| \le 4$) is weaker than gravity. The running
cost is periodic in the angle, zero upright and two hanging:

$$g(x, u) = 1 + \cos\theta + 0.01\,\dot\theta^2 + 0.01\,u^2,$$

written with `jax.numpy` so that it traces inside the compiled rollouts.

In [ ]:
import jax.numpy as jnp
import numpy as np

from minilink import CostFunction, Pendulum

plant = Pendulum()
plant.inputs["u"].lower = np.array([-4.0])
plant.inputs["u"].upper = np.array([4.0])
plant.state.lower = np.array([-4.0 * np.pi, -20.0])
plant.state.upper = np.array([4.0 * np.pi, 20.0])


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta = x[0]
        dtheta = x[1]

        # 1 + cos θ is zero at the upright; a little speed and a little torque
        g = 1.0 + jnp.cos(theta) + 0.01 * dtheta**2 + 0.01 * u[0] ** 2

        return g

    def h(self, x, t=0.0, params=None):
        return 0.0


cost = SwingUpCost()

The problem has an infinite horizon, `horizon=np.inf`, and its start is a law: `Uniform` over
the whole circle with small rates. `sample_x0` draws from it.

In [ ]:
from minilink import PlanningProblem, Uniform

problem = PlanningProblem(
    plant,
    cost=cost,
    horizon=np.inf,
    x0=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
    U=plant.input_box,
)
problem.sample_x0(0, n=3)

`ReinforcementLearningPlanner` trains with PPO on 64 simulated pendulums in parallel. The
network sees features of the state, $(\cos\theta, \sin\theta, 0.1\,\dot\theta)$, which
make the angle periodic; the law is still a function of $x$. The discount $\gamma = 0.97$
per control period of 0.05 s weighs about the next 1.6 s, the length of one swing.
Training takes a few seconds.

In [ ]:
from minilink import ReinforcementLearningPlanner


def features(x):
    theta = x[0]
    dtheta = x[1]

    return jnp.array([jnp.cos(theta), jnp.sin(theta), 0.1 * dtheta])


planner = ReinforcementLearningPlanner(
    problem,
    dt=0.05,
    features=features,
    hidden=(32, 32),
    method="ppo",
    n_rollouts=64,
    n_steps_per_update=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
)
solution = planner.solve(n_timesteps=120_000)
print(solution.record)

In [ ]:
planner.plot_learning_curve();

## 2. The policy is a controller block

`planner.controller` is the learned law as a `NeuralPolicy`, an ordinary `System`: it
maps the state to a torque within the motor's bounds. `controller @ plant` closes the loop, and
the loop simulates like any other. From hanging, the law swings once each way and comes to
rest near the upright at $\theta \approx -\pi$, the same position one turn away. A small
steady offset remains: nothing in the learned law integrates the error.

In [ ]:
controller = planner.controller

plant.x0 = np.array([0.05, 0.0])
closed = controller @ plant
traj = closed.compute_trajectory(tf=10.0, dt=0.01)
closed.plot_trajectory(traj)

Its law can be drawn over the state space, like the value-iteration law of
`02_optimal_control`.

In [ ]:
controller.plot_control_law(x_axis=0, y_axis=1, u_axis=0)

`evaluate` scores any controller on the stochastic problem by Monte Carlo: 100 starts
drawn from the law, the same draws for every controller (`key`). The baseline is a
state-feedback law about the upright, $u = -K(x - x^\star)$; it asks for far more torque
than the motor has, so it is almost never feasible. The learned law is feasible from every
start, at a fraction of the cost.

In [ ]:
from minilink import StateFeedback, evaluate

x_up = np.array([np.pi, 0.0])
baseline = StateFeedback(K=[[30.0, 8.0]], x_bar=x_up)

runs = dict(dt=0.05, n_trials=100, episode_length=10.0, key=1)

print(evaluate(problem, baseline, **runs))
print(evaluate(problem, controller, **runs))

## 3. What the learner sees

The planner compiles the plant with JAX and wraps it into an environment, `planner.env`.
One `step` is one control period: the dynamics with the input held, and the reward
$r_k = -g(x_k, u_k)\,\Delta t$, the running cost with its sign flipped, so maximizing the
return minimizes $J$. An episode ends *terminated* when its cost is fully counted, or
*truncated* when the state leaves the plant's box (the critic's estimate stands in for the
rest).

In [ ]:
print(planner.env.describe())

The same planner trains off-policy with `method="sac"` (Soft Actor-Critic): fewer plant
steps, more computation per step. Everything around it, the problem, the evaluation and
the controller block, stays the same.

## 4. A learned law is analyzed like any other

The closed loop is a diagram, so the analysis tools apply to the learned law with no
adapter. Linearizing at the upright differentiates through the network and the plant; the
eigenvalues of $A$ in the left half-plane say that the network stabilizes the top
locally.

In [ ]:
linear_loop = closed.linearize(x_up)
np.linalg.eigvals(linear_loop.A())

## 5. A bigger plant: learning to fly

The same four steps on a planar drone: six states, two thrusters, and a start drawn from a
`Gaussian` law up to 5 m away and tilted. The plant is the catalog's `Drone2D` with its two
thrusts normalized, so that $u = 0$ hovers and $u = \pm 1$ is the least and the most each
thruster gives. The cost is quadratic about the hover at the origin.

In [ ]:
from minilink import Drone2D, Gaussian


class NormalizedDrone(Drone2D):
    def __init__(self):
        super().__init__()

        # Parameters: a 1 kg drone, its thrusters giving up to 1.2 times its weight
        self.params["mass"] = 1.0
        self.params["inertia"] = 0.1
        self.thrust_to_weight = 1.2

        # The inputs are normalized thrusts; the states an episode may visit
        self.inputs["u"].lower = np.array([-1.0, -1.0])
        self.inputs["u"].upper = np.array([1.0, 1.0])
        self.state.upper = np.array([10.0, 10.0, 2.0 * np.pi, 10.0, 10.0, 10.0])
        self.state.lower = -self.state.upper

    def thrusts(self, u, params):
        mass = params["mass"]
        gravity = params["gravity"]
        ratio = self.thrust_to_weight

        # T = m g (½ + (ratio − 1) u): half the weight each at u = 0
        T = mass * gravity * (0.5 + (ratio - 1.0) * u)

        return T

    def f(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params

        dx = super().f(x, self.thrusts(u, params), t, params)

        return dx

    def get_dynamic_geometry(self, x, u, t=0, params=None):
        params = self.params if params is None else params

        return super().get_dynamic_geometry(x, self.thrusts(u, params), t, params)


class HoverCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        Q = jnp.diag(jnp.array([1.0, 1.0, 6.0, 0.1, 0.1, 0.1]))
        R = 0.001 * jnp.eye(2)

        g = x @ Q @ x + u @ R @ u

        return g

    def h(self, x, t=0.0, params=None):
        return 0.0


drone = NormalizedDrone()
flight = PlanningProblem(
    drone,
    cost=HoverCost(),
    horizon=np.inf,
    x0=Gaussian(np.zeros(6), [5.0, 5.0, 1.0, 1.0, 1.0, 0.2]),
    U=drone.input_box,
)

Training takes about ten seconds. The learned law is scored on 100 starts drawn from the
problem's law, then flown from one of them. It brings the drone back from most starts; the
mean cost and its large spread come from the few that start far away and tilted, which this
short training does not master.

In [ ]:
pilot = ReinforcementLearningPlanner(
    flight,
    dt=0.05,
    normalize=False,
    n_rollouts=1,
    n_steps_per_update=2048,
    batch_size=64,
    gamma=0.99,
)
pilot.solve(n_timesteps=150_000)

print(evaluate(flight, pilot.controller, **runs))

In [ ]:
drone.x0 = np.array([-1.0, -2.0, 1.0, 0.0, 0.0, 0.0])
flight_loop = pilot.controller @ drone
flight_traj = flight_loop.compute_trajectory(tf=10.0, dt=0.01)
flight_loop.plot_trajectory(flight_traj)
flight_traj.x[:, -1].round(2)

In [ ]:
flight_loop.animate(flight_traj)

On a robot arm the same steps train a set-point law over an impedance loop, and the learned
loop is read with a Bode plot: `examples/demos/rl/ur5_rl_to_bode.py`.

## Summary

| Step | Code |
| --- | --- |
| A stochastic problem | `PlanningProblem(plant, cost=..., horizon=np.inf, x0=Uniform(...))` |
| Learn | `ReinforcementLearningPlanner(problem, dt=..., method="ppo").solve(n_timesteps=...)` |
| The policy as a block | `controller = planner.controller`; `controller @ plant` |
| Score by Monte Carlo | `evaluate(problem, controller, dt=..., n_trials=..., key=...)` |
| Analyze | `closed.linearize(x_up)` |

Next, [`05_differentiable`](05_differentiable.ipynb): gradients through whole loops.